# FRET: donor, acceptor, and the time-resolved efficiency

Two dyes on the same protein. QuEst simulates the donor's diffusion and
evaluates the transfer rate **per trajectory frame**, so the FRET channel
responds to where the dye actually is.

Reproduces the FRET part of the old `simulate_decays_for_structure` notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "quest_notebook.py").exists() else Path.cwd() / "notebooks"))
from quest_notebook import REPO, STRUCTURE, quick_project, fret_project

import numpy as np
import matplotlib.pyplot as plt

print("structure:", STRUCTURE.name)

In [ ]:
from quest.api import simulate_project

# R0 is in Angstrom, like every length in a project. A value in
# nanometres (~5.5) is ten times too small and switches FRET off silently.
pair = fret_project(donor=55, acceptor=132)
result = simulate_project(pair, save_outputs=False)

print(f"donor QY            {result.quantum_yield_donor:.4f}")
print(f"donor QY with A     {result.quantum_yield_fret:.4f}")
print(f"transfer efficiency {result.fret_efficiency:.4f}")


## Both decays, and E(t)

The FRET channel decays faster; the ratio of the two is the time-resolved
transfer efficiency, which is what the E(t) analysis works on.

In [ ]:
t = result.time
fd0 = np.clip(result.donor_counts.astype(float), 1e-9, None)
fda = np.clip(result.fret_counts.astype(float), 1e-9, None)

fig, (ax, bx) = plt.subplots(1, 2, figsize=(11, 4))
ax.semilogy(t, fd0 / fd0.max(), label="donor only")
ax.semilogy(t, fda / fda.max(), label="donor + acceptor")
ax.set_xlim(0, 25); ax.set_xlabel("time [ns]"); ax.set_ylabel("normalised counts")
ax.legend(); ax.set_title("Decays")

valid = (fd0 > 1.0) & (t < 20)
bx.plot(t[valid], 1.0 - fda[valid] / fd0[valid])
bx.set_xlabel("time [ns]"); bx.set_ylabel("E(t)"); bx.set_ylim(0, 1)
bx.set_title("Time-resolved transfer efficiency")
plt.tight_layout()

## Efficiency responds to the Förster radius

A sanity check that costs one more run: a smaller R0 must transfer less.

In [ ]:
weak = simulate_project(fret_project(55, 132, r0=35.0), save_outputs=False)
print(f"R0 = 52 A -> E = {result.fret_efficiency:.3f}")
print(f"R0 = 35 A -> E = {weak.fret_efficiency:.3f}")
assert weak.fret_efficiency < result.fret_efficiency
